<a href="https://colab.research.google.com/github/lichkkkk/ml_coding/blob/main/rl/GSM8K_GRPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Repo setup

In [ ]:
!git clone https://github.com/stanford-cs336/assignment5-alignment.git

In [ ]:
%cd assignment5-alignment
!uv pip install --system -r pyproject.toml

# Data Setup

In [ ]:
import json

_GSM8K_TRAIN_PATH = 'data/gsm8k/train.jsonl'
_GSM8K_TEST_PATH = 'data/gsm8k/test.jsonl'

def _load_gsm8k(path: str) -> tuple[list[str], list[str]]:
    '''
    return list of (question, ground truth pair)
    '''
    questions = []
    answers = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                example = json.loads(line)
                questions.append(example['question'])
                answers.append(example['answer'])
    return questions, answers


def load_gsm8k_train():
    return _load_gsm8k(_GSM8K_TRAIN_PATH)


def load_gsm8k_test():
    return _load_gsm8k(_GSM8K_TEST_PATH)


def extract_gsm8k_gt(answer: str) -> str:
    if "####" in answer:
        return answer.split("####")[-1].strip()
    return answer.strip()

# Device / GPU

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(DEVICE)

if torch.cuda.is_available():
    device_id = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_id)

    print(f"Device Name: {torch.cuda.get_device_name(device_id)}")
    print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
    print(f"Allocated:    {torch.cuda.memory_allocated(device_id) / 1024**3:.2f} GB")
    print(f"Reserved:     {torch.cuda.memory_reserved(device_id) / 1024**3:.2f} GB")
else:
    print("CUDA is not available. Using CPU.")

# Sampling

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def get_model_and_tokenizer(model_id_or_dir: str, device: str):
    if device == "cpu":
        dtype = torch.float32
        attn_impl = "eager"
        device_map = None
    else:
        # Safer default for Colab.
        # T4 does not properly support bf16, so fp16 is safer.
        major, _ = torch.cuda.get_device_capability()
        dtype = torch.bfloat16 if major >= 8 else torch.float16
        attn_impl = "sdpa"
        device_map = {"": device}

    model = AutoModelForCausalLM.from_pretrained(
        model_id_or_dir,
        device_map=device_map,
        torch_dtype=dtype,
        attn_implementation=attn_impl,
    )

    if device == "cpu":
        model = model.to(device)

    tokenizer = AutoTokenizer.from_pretrained(model_id_or_dir)
    return model, tokenizer

MODEL = 'allenai/OLMo-2-0425-1B'
model, tokenizer = get_model_and_tokenizer(MODEL, DEVICE)

In [ ]:
from typing import Any, Literal
import numpy as np
from transformers import PreTrainedModel, PreTrainedTokenizer
from cs336_alignment import drgrpo_grader
from tqdm.notebook import tqdm


_QUESTION_ONLY_TEMPLATE = '''{question} Please put your final answer within \\boxed{{}}.'''
_R1_ZERO_TEMPLATE = '''A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>'''
_R1_ZERO_THREE_SHOT_TEMPLATE = '''A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?
Assistant: <think> There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. So the answer is 6. </think> <answer> 6 </answer>
User: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?
Assistant: <think> There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. So the answer is 5. </think> <answer> 5 </answer>
User: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?
Assistant: <think> Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. So the answer is 39. </think> <answer> 39 </answer>
User: {question}
Assistant: <think>'''

TEMPLATE_MAP = {
    'question_only': _QUESTION_ONLY_TEMPLATE,
    'r1_zero': _R1_ZERO_TEMPLATE,
    'r1_zero_three_shot': _R1_ZERO_THREE_SHOT_TEMPLATE
}


def build_prompt(
    question: str,
    template: Literal['r1_zero', 'r1_zero_three_shot', 'question_only'],
) -> str:
    template_str = TEMPLATE_MAP[template]
    return template_str.format(question=question)


@torch.no_grad()
def sample(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    prompts: list[str],
    max_input_len: int = 512,
    max_output_len: int = 512,
    n_samples = 1,
    stop_string: str = r"</answer>"
) -> list[list[str]]:
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    batch = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_input_len,
    )
    batch = {k: v.to(model.device) for k, v in batch.items()}

    responses = []

    for _ in range(n_samples):
      outputs = model.generate(
          input_ids=batch["input_ids"],
          attention_mask=batch["attention_mask"],
          max_new_tokens=max_output_len,
          do_sample=True,
          temperature=1,
          pad_token_id=tokenizer.pad_token_id,
          # for early stop
          stop_strings=[stop_string],
          tokenizer=tokenizer,
      )
      input_len = batch["input_ids"].shape[1]
      generated_ids = outputs[:, input_len:]
      texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
      responses.append(texts)

    # transpose responses
    responses = np.transpose(responses)
    return responses  # (prompts, n_samples)


def grade_responses(responses: list[str], ground_truth: str) -> tuple[dict[str, float], dict[str, float]]:
  '''
  return pass@k rewards where k = len(responses)
  '''
  assert len(responses) > 0 and isinstance(responses[0], str)  # drgrpo_grader.r1_zero_reward_fn does not throw error if wrong obj is passed
  raw_rewards = [drgrpo_grader.r1_zero_reward_fn(r, ground_truth) for r in responses]
  agg_rewards = {}
  for k in raw_rewards[0].keys():
    agg_rewards[k] = np.max([r[k] for r in raw_rewards])
  return agg_rewards, raw_rewards

In [ ]:
# test sample

test_questions, test_answers = load_gsm8k_test()
test_question, test_answer = test_questions[0], test_answers[0]
test_prompt = build_prompt(test_question, 'r1_zero_three_shot')
print('TEST_PROMPT: ', test_prompt)
print('TEST_ANSWER: ', test_answer)

test_responses = sample(model, tokenizer, [test_prompt], n_samples=2)
for r in test_responses[0]:
  print('TEST_RESPONSE:\n', r)

test_ground_truth = extract_gsm8k_gt(test_answer)
print('REWARD: ', grade_responses(test_responses[0], test_ground_truth))

# Baseline

In [ ]:
def eval_gsm8k(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    eval_size: int,
    eval_prompt_temaple: Literal['r1_zero', 'r1_zero_three_shot', 'question_only'],
    eval_batch_size: int = 32,
    k = 1,  # pass@k
) -> tuple[dict[str, float], dict[str, Any]]:
    test_questions, test_answers = load_gsm8k_test()
    eval_questions, eval_answers = test_questions[:eval_size], test_answers[:eval_size]

    eval_prompts = [build_prompt(q, eval_prompt_temaple) for q in eval_questions]
    sampled_responses = []  # (prompts, n_samples)
    for i in tqdm(range(0, len(eval_prompts), eval_batch_size)):
        eval_prompts_batch = eval_prompts[i:i+eval_batch_size]
        sampled_responses_batch = sample(model, tokenizer, eval_prompts_batch, n_samples=k)
        sampled_responses.extend(sampled_responses_batch)

    ground_truths = [extract_gsm8k_gt(a) for a in eval_answers]
    format_rewards, answer_rewards, rewards = [], [], []
    debug_raw_rewards = []
    for r, gt in zip(sampled_responses, ground_truths):
        agg_rewards, raw_rewards = grade_responses(r, gt)
        format_rewards.append(agg_rewards['format_reward'])
        answer_rewards.append(agg_rewards['answer_reward'])
        rewards.append(agg_rewards['reward'])
        debug_raw_rewards.append(raw_rewards)

    res_dict = {
        'format_reward': np.mean(format_rewards),
        'answer_reward': np.mean(answer_rewards),
        'reward': np.mean(rewards)
    }
    debug_dict = {
        'format_rewards': format_rewards,
        'answer_rewards': answer_rewards,
        'rewards': rewards,
        'debug_raw_rewards': debug_raw_rewards,
        'eval_prompts': eval_prompts,
        'sampled_responses': sampled_responses,
        'ground_truths': ground_truths,
    }
    return res_dict, debug_dict

## test eval

EVAL_SIZE = 16
PROMPT_TEMPLATE = 'r1_zero'
PASS_AT_K = 16

res, debug = eval_gsm8k(
    model,
    tokenizer,
    eval_size=EVAL_SIZE,
    eval_prompt_temaple=PROMPT_TEMPLATE,
    k=PASS_AT_K,
)

print(res)

In [ ]:
# # manual inspect
# i = 0
# print(debug['eval_prompts'][i])
# print(debug['sampled_responses'][i])
# print(debug['ground_truths'][i])
# print(debug['answer_rewards'][i])

In [ ]:
# baseline sweep
'''
#### r1_zero
pass@1: {'format_reward': np.float64(0.453125), 'answer_reward': np.float64(0.0), 'reward': np.float64(0.0)}
pass@4: {'format_reward': np.float64(0.9375), 'answer_reward': np.float64(0.0), 'reward': np.float64(0.0)}
pass@16: {'format_reward': np.float64(1.0), 'answer_reward': np.float64(0.015625), 'reward': np.float64(0.015625)}

#### r1_zero_three_shots
pass@1: {'format_reward': np.float64(0.9609375), 'answer_reward': np.float64(0.21875), 'reward': np.float64(0.21875)}
pass@4: {'format_reward': np.float64(1.0), 'answer_reward': np.float64(0.375), 'reward': np.float64(0.375)}
pass@16: {'format_reward': np.float64(1.0), 'answer_reward': np.float64(0.6640625), 'reward': np.float64(0.6640625)}
'''

import itertools

eval_size = 128
prompt_templates = ['r1_zero_three_shot']
k = [1, 4, 16]

for prompt_template, k in itertools.product(prompt_templates, k):
  res, _ = eval_gsm8k(
      model,
      tokenizer,
      eval_size=eval_size,
      eval_prompt_temaple=prompt_template,
      eval_batch_size=128,
      k=k,
  )
  print(prompt_template, k, res)

# GRPO Impl

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.optim import Optimizer
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PreTrainedTokenizer
from typing import Callable, Literal
import numpy as np


def get_model_and_tokenizer(model_id_or_dir: str, device: str):
    model = AutoModelForCausalLM.from_pretrained(
        model_id_or_dir,
        device_map=device,
        dtype=torch.bfloat16,
        attn_implementation='eager' if device=='cpu' else 'flash_attention_2',
    )
    tokenizer=AutoTokenizer.from_pretrained(model_id_or_dir)
    return model, tokenizer


def tokenize_prompt_and_output(
    prompt_strs: list[str],
    output_strs: list[str],
    tokenizer: PreTrainedTokenizer
) -> dict[str, torch.Tensor]:
    '''
    Returns:
        dict[str, torch.Tensor] with following keys:
          - input_ids: ids w/ shape (B, max_len - 1)
          - labels: ids w/ shape (B, max_len - 1) (input shifted by 1)
          - response_mask: bool mask w/ shape (B, max_len - 1), True for label pos
    '''
    assert len(prompt_strs) == len(output_strs), 'length not equal'
    input_ids, labels, masks = [], [], []
    for i in range(len(prompt_strs)):
        prompt_ids = tokenizer(prompt_strs[i], return_tensors='pt', return_token_type_ids=False)['input_ids']
        output_ids = tokenizer(output_strs[i], return_tensors='pt', return_token_type_ids=False)['input_ids']
        full_ids = torch.cat([prompt_ids, output_ids]).reshape(-1)

        input_ids.append(full_ids)
        labels.append(full_ids)

        mask = torch.cat([torch.zeros_like(prompt_ids), torch.ones_like(output_ids)]).reshape(-1)
        masks.append(mask)

    padded_input_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)  # (B, max_len)
    padded_labels = pad_sequence(labels, batch_first=True, padding_value=0)
    padded_masks = pad_sequence(masks, batch_first=True, padding_value=0)

    shifted_input_ids = padded_input_ids[:, :-1]
    shifted_labels = padded_labels[:, 1:]
    shifted_masks = padded_masks[:, 1:]

    return {
        'input_ids': shifted_input_ids,
        'labels': shifted_labels,
        'response_mask': shifted_masks
    }


def get_response_log_probs(
    model: PreTrainedModel,
    input_ids: torch.Tensor,
    labels: torch.Tensor,
    return_token_entropy: bool = False
) -> dict[str, torch.Tensor]:
    '''
    Args:
        model: model
        input_ids: (B, S)
        labels: (B, S)
        return_token_entropy: ?
    Returns:
        dict[str, torch.Tensor] with keys:
            - log_probs: (B, S) log probs for the given correct labels
            - token_entropy: (B, S) per-token entropy
    '''
    logits = model(input_ids).logits  # (B, S, V)
    logprobs = F.log_softmax(logits, dim=-1)  # (B, S, V)
    label_logprobs = torch.gather(logprobs, dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)  # (B, S)
    outputs = {
        'log_probs': label_logprobs
    }
    if return_token_entropy:
        probs = logprobs.exp()  # (B, S, V)
        entropy = - (probs * logprobs).sum(dim=-1)  # (B, S)
        outputs['token_entropy'] = entropy
    return outputs


def compute_rollout_rewards(
    reward_fn: Callable[[str, str], dict[str, float]],
    rollout_responses: list[str],
    repeated_ground_truths: list[str]
) -> tuple[torch.Tensor, dict[str, float]]:
    '''
    Args:
        - reward_fn: score response against GT, outputs 'reward', 'format_reward', 'answer_reward'
        - rollout_responses: (B*G) shape list of respones
        - repeated_ground_truths: (B*G) shape ground truth
    Returns:
        - tuple[torch.Tensor, dict[str, float]]:
            - raw_rewards: (B*G) size
            - metadata for logging: mean total reward, mean format reward, etc.
    '''
    outputs = {
        'reward': [],
        'format_reward': [],
        'answer_reward':  []
    }
    for resp, gt in zip(rollout_responses, repeated_ground_truths):
        rewards = reward_fn(resp, gt)
        for r, v in rewards.items():
            outputs[r].append(v)

    raw_rewards = torch.tensor(outputs['reward'])
    metadata = {}
    metadata['mean_reward'] = np.mean(outputs['reward'])
    metadata['mean_format_reward'] = np.mean(outputs['format_reward'])
    return raw_rewards, metadata


def compute_group_normalized_rewards(
    raw_rewards: torch.Tensor,
    group_size: int,
    baseline: Literal['mean', 'none'] = 'mean',
    advantage_eps: float = 1e-6,
    advantage_normalizer: Literal['std', 'none', 'mean'] = 'std',
) -> tuple[torch.Tensor, dict[str, float]]:
    '''
    Args:
        - raw_rewards: (B*G,) float rewards
    Returns:
        - tuple[torch.Tensor, dict[str, float]]:
            - advantages: (B*G,)
            - metadata to log
    '''
    metadata = {}
    adv = raw_rewards.reshape(-1, group_size)  # (B, G)
    adv_mean = adv.mean(dim=-1, keepdim=True)  # (B, 1)

    if baseline == 'mean':
        adv = adv - adv_mean

    if advantage_normalizer == 'std':
        adv_std = adv.std(dim=-1, keepdim=True)  # (B, 1)
        adv = adv / (adv_std + advantage_eps)
        metadata['adv_std_mean'] = adv_std.mean().item()
    elif advantage_normalizer == 'mean':
        adv = adv / (adv_mean + advantage_eps)
        metadata['adv_mean_mean'] = adv_mean.mean().item()

    adv = adv.reshape(-1)
    return adv, metadata


def compute_policy_gradient_loss(
    raw_rewards_or_advantages: torch.Tensor,
    policy_log_probs: torch.Tensor,
    importance_reweighting_method: Literal['none', 'noclip', 'grpo', 'gspo'] = 'none',
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
    response_mask: torch.Tensor | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    '''
    Args:
        - raw_rewards_or_advantages: (B,)
        - importance_reweighting_method
        - policy_log_probs: (B, S)
        - old_log_probs: (B, S)
        - cliprange
        - response_mask: (B, S)

    Returns:
        - tuple[torch.Tensor, dict[str, torch.Tensor]]
            - per_token_policy_gradient_loss: (B, S) - to be aggregated during training
            - metadata: stats like clip-fraction components
    '''
    metadata = {}
    B = raw_rewards_or_advantages.shape[0]
    adv = raw_rewards_or_advantages.reshape(B, 1)  # (B, 1)

    if importance_reweighting_method == 'none':
        per_token_loss = - policy_log_probs * adv

    elif importance_reweighting_method == 'noclip':
        ratio = (policy_log_probs - old_log_probs).exp()
        per_token_loss = - ratio * adv

        metadata['ratio'] = ratio.detach()
    elif importance_reweighting_method == 'grpo':
        assert cliprange is not None
        ratio = (policy_log_probs - old_log_probs).exp()
        loss1 = ratio * adv
        loss2 = torch.clamp(ratio, min=1-cliprange, max=1+cliprange) * adv
        per_token_loss = - torch.minimum(loss1, loss2)

        metadata['ratio'] = ratio.detach()
    elif importance_reweighting_method == 'gspo':
        assert response_mask is not None
        log_ratio = policy_log_probs - old_log_probs  # (B, S)
        seq_log_ratio = (log_ratio * response_mask).sum(dim=-1) / response_mask.sum(dim=-1)  # (B,)
        seq_ratio = seq_log_ratio.exp().unsqueeze(-1)  # (B, 1)
        loss1 = seq_ratio * adv  # (B, S)
        loss2 = torch.clamp(seq_ratio, min=1-cliprange, max=1+cliprange) * adv
        per_token_loss = - torch.minimum(loss1, loss2)

        metadata['ratio'] = seq_ratio.detach()
    else:
        raise ValueError('bad method')

    return per_token_loss, metadata


def aggregate_loss_across_microbatch(
    per_token_policy_gradient_loss: torch.Tensor,
    mask: torch.Tensor,
    loss_normalization: Literal['sequence', 'constant'] = 'sequence',
    normalization_constant: int | None = None,
) -> torch.Tensor:
    '''
    Args:
        - per_token_policy_gradient_loss: (B, S)
        - mask: (B, S)
        - loss_normalization
        - normalization_constant
    Retunrs:
        - loss which can call backword
    '''
    loss = per_token_policy_gradient_loss * mask  # (B, S)
    if loss_normalization == 'sequence':
        loss = loss.sum(dim=-1) / mask.sum(dim=-1)
        loss = loss.mean(dim=-1)
    elif loss_normalization == 'constant':
        loss = loss.reshape(-1).sum() / normalization_constant
    else:
        raise ValueError('bad normalization')
    return loss


def grpo_train_step(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    optimizer: Optimizer,
    gradient_accumulation_steps: int,
    max_grad_norm: float | None,
    reward_fn: Callable[[str, str], dict[str, float]],
    repeated_prompts: list[str],
    rollout_responses: list[str],
    repeated_ground_truths: list[str],
    group_size: int,
    # Reward normalization
    baseline: Literal['mean', 'none'] = 'mean',
    advantage_eps: float = 1e-6,
    advantage_normalizer: Literal['std', 'none', 'mean'] = 'std',
    # Importance reweighting and clipping
    importance_reweighting_method: Literal['none', 'noclip', 'grpo', 'gspo'] = 'none',
    old_log_probs: torch.Tensor | None = None,
    cliprange: float | None = None,
    # Loss normalization
    loss_normalization: Literal['sequence', 'constant'] = 'sequence',
    normalization_constant: int | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor | str]]:
    '''
    One compelte training step given rollouts
    '''
    metadata = {}

    # Step 1: calculate advs based on rollouts
    raw_rewards, _metadata = compute_rollout_rewards(
        reward_fn,
        rollout_responses,
        repeated_ground_truths
    )
    metadata.update(_metadata)

    advs, _metadata = compute_group_normalized_rewards(
        raw_rewards,
        group_size,
        baseline,
        advantage_eps,
        advantage_normalizer,
    )
    metadata.update(_metadata)

    # Step 2: forward with policy
    tokenizer_outputs = tokenize_prompt_and_output(
        repeated_prompts,
        rollout_responses,
        tokenizer
    )
    input_ids = tokenizer_outputs['input_ids']  # (B, S)
    labels = tokenizer_outputs['labels']
    response_mask = tokenizer_outputs['response_mask']

    ## micro batch
    batch_size = input_ids.shape[0]
    microbatch_size = batch_size // gradient_accumulation_steps

    losses = []
    for i in range(0, batch_size, microbatch_size):
        microbatch_advs = advs[i:i+microbatch_size]
        microbatch_input_ids = input_ids[i:i+microbatch_size]
        microbatch_labels = labels[i:i+microbatch_size]
        microbatch_response_mask = response_mask[i:i+microbatch_size]
        microbatch_old_log_probs = old_log_probs[i:i+microbatch_size] if old_log_probs is not None else None

        forward_outputs = get_response_log_probs(
            model,
            microbatch_input_ids,
            microbatch_labels,
            return_token_entropy= False
        )
        microbatch_policy_log_probs = forward_outputs['log_probs']

        per_token_policy_gradient_loss, _metadata = compute_policy_gradient_loss(
            microbatch_advs,
            microbatch_policy_log_probs,
            importance_reweighting_method,
            microbatch_old_log_probs,
            cliprange,
            microbatch_response_mask,
        )
        metadata.update({f'{k}_{i}': v for k, v in _metadata.items()})

        loss = aggregate_loss_across_microbatch(
            per_token_policy_gradient_loss,
            microbatch_response_mask,
            loss_normalization,
            normalization_constant,
        )
        loss = loss * (len(microbatch_advs) / len(advs))  # loss is mean over batch

        loss.backward()
        losses.append(loss.detach())


    # Step 3: udpate params
    if max_grad_norm is not None:
        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=max_grad_norm
        )

    optimizer.step()
    optimizer.zero_grad()

    batch_loss = torch.stack(losses).sum()
    return batch_loss, metadata


# Train

In [ ]:
TRAINING_STEPS = 1

